# Wave simulations: with and without viscous damping

Dit notebook runt per golfconditie twee OrcaFlex-simulaties:

- zonder viskeuze damping
- met viskeuze damping

Daarna worden heave, roll en pitch geplot. De plots worden alleen getoond en niet opgeslagen.


In [1]:
# ============================================================
# 1. Imports
# ============================================================
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import OrcFxAPI


In [2]:
# ============================================================
# 2. Input: paden en objectnamen
# ============================================================
# PAS AAN naar jouw OrcaFlex modelbestand
model_path = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Harlequin_spring_120s_withmooring.dat")

# Namen zoals in jouw eerdere notebook
vesselname = "floaters"
vesseltypename = "floatertype"

# Simulatieduur [s]
# simulation_duration = 300.0

# Starttijd voor plotten [s]
# Zet bijvoorbeeld op 60 als je de opstarttransient niet wilt zien.
plot_start_time = 0.0


In [3]:
# # ============================================================
# # 3. Golfcondities
# # ============================================================
# wave_conditions = pd.DataFrame(
#     [
#         {"Hs": 1.0, "Tp": 4.0},
#         {"Hs": 2.0, "Tp": 6.0},
#         {"Hs": 4.0, "Tp": 8.0},
#         {"Hs": 4.0, "Tp": 10.0},
#         {"Hs": 4.0, "Tp": 12.0},
#         {"Hs": 4.0, "Tp": 14.0},
#         {"Hs": 4.0, "Tp": 16.0},
#         {"Hs": 4.0, "Tp": 7.04},
#         {"Hs": 2.0, "Tp": 7.04},
#         {"Hs": 1.0, "Tp": 4.15},
#         {"Hs": 2.0, "Tp": 4.15},

#     ]
# )


# wave_conditions = pd.DataFrame(
#     [
        
#         {"Hs": 1.0, "Tp": 7.04},
#         {"Hs": 2.0, "Tp": 7.04},
#         {"Hs": 3.0, "Tp": 7.04},
#         {"Hs": 2.0, "Tp": 7.04},
#         {"Hs": 4.0, "Tp": 7.04},
#         {"Hs": 5.0, "Tp": 7.04},
#         {"Hs": 6.0, "Tp": 7.04},
#         {"Hs": 7.0, "Tp": 7.04},

#         {"Hs": 1.0, "Tp": 7.03},
#         {"Hs": 2.0, "Tp": 7.03},
#         {"Hs": 3.0, "Tp": 7.03},
#         {"Hs": 2.0, "Tp": 7.03},
#         {"Hs": 4.0, "Tp": 7.03},
#         {"Hs": 5.0, "Tp": 7.03},
#         {"Hs": 6.0, "Tp": 7.03},
#         {"Hs": 7.0, "Tp": 7.03},
    

#     ]
# )

# wave_conditions


In [4]:
# ============================================================
# 4. Viskeuze dampingwaardes
# ============================================================
viscous_damping = {
    "heave": {"lin": 1.840277778, "quad": 2.756944444},
    "pitch": {"lin": 253.9814815, "quad": 564.3518519},
    "roll":  {"lin": 227.3148148, "quad": 317.5},
}

viscous_damping


{'heave': {'lin': 1.840277778, 'quad': 2.756944444},
 'pitch': {'lin': 253.9814815, 'quad': 564.3518519},
 'roll': {'lin': 227.3148148, 'quad': 317.5}}

In [5]:
# ============================================================
# 5. Helper functies: golfconditie en damping instellen
# ============================================================
def set_wave_condition(model, Hs: float, Tp: float):
    """
    Zet de golfconditie in het OrcaFlex model.
    Controleer eventueel de veldnamen als jouw model een andere wave setup gebruikt.
    """
    env = model.environment

    env.WaveType = "JONSWAP"
    env.WaveHs = float(Hs)
    env.WaveTz= float(Tp)

    # Optioneel zelf aanzetten indien nodig:
    # env.WaveDirection = 0.0
    # env.WaveSeed = 1
    # env.WaveGamma = 3.3


# def set_simulation_duration(model, duration: float):
#     """
#     Zet de simulatieduur.
#     """
#     try:
#         model.general.StageDuration = [float(duration)]
#     except Exception:
#         model.general.StageDuration[0] = float(duration)


def set_damping(vesseltype, dof: str, lin_coeff: float, quad_coeff: float):
    """
    Zelfde veldnamen/stijl als in je decay-notebook.
    Pas dit blok eventueel aan naar jouw exacte OrcaFlex-veldnamen.
    """
    dof = dof.lower()

    if dof == "pitch":
        vesseltype.OtherDampingLinearCoeffRy = float(lin_coeff)
        vesseltype.OtherDampingQuadraticCoeffRy = float(quad_coeff)

    elif dof == "roll":
        vesseltype.OtherDampingLinearCoeffRx = float(lin_coeff)
        vesseltype.OtherDampingQuadraticCoeffRx = float(quad_coeff)

    elif dof == "heave":
        vesseltype.OtherDampingLinearCoeffz = float(lin_coeff)
        vesseltype.OtherDampingQuadraticCoeffz = float(quad_coeff)

    else:
        raise ValueError(f"Onbekende DOF: {dof}")


def set_all_viscous_damping(vesseltype, use_viscous_damping: bool):
    """
    Zet heave, roll en pitch damping tegelijk aan of uit.
    """
    for dof in ["heave", "roll", "pitch"]:
        if use_viscous_damping:
            lin = viscous_damping[dof]["lin"]
            quad = viscous_damping[dof]["quad"]
        else:
            lin = 0.0
            quad = 0.0

        set_damping(vesseltype, dof, lin, quad)


In [6]:
# ============================================================
# 6. Helper functies: simulatie runnen en response uitlezen
# ============================================================
def run_orcaflex_wave_simulation(model_path: Path, Hs: float, Tp: float, use_viscous_damping: bool):
    """
    Laadt model, stelt golfconditie en damping in, runt simulatie en geeft model terug.
    """
    model = OrcFxAPI.Model(str(model_path))

    vessel = model[vesselname]
    vesseltype = model[vesseltypename]

    # Zorg dat eventuele initiële offset/rotatie uit decay-tests uit staat
    vessel.InitialX = 0.0
    vessel.InitialY = 0.0
    vessel.InitialZ = 0.0
    vessel.InitialHeel = 0.0
    vessel.InitialTrim = 0.0
    vessel.InitialHeading = 0.0

    # set_wave_condition(model, Hs, Tp)
    # set_simulation_duration(model, simulation_duration)
    set_all_viscous_damping(vesseltype, use_viscous_damping)

    model.RunSimulation()
    return model


def get_wave_response(model):
    """
    Leest tijd, heave, roll en pitch uit het OrcaFlex model.
    """
    vessel = model[vesselname]

    t = np.asarray(model.general.TimeHistory("Time"), dtype=float)
    heave = np.asarray(vessel.TimeHistory("Z"), dtype=float)
    roll = np.asarray(vessel.TimeHistory("Rotation 1"), dtype=float)
    pitch = np.asarray(vessel.TimeHistory("Rotation 2"), dtype=float)

    return {
        "time": t,
        "heave": heave,
        "roll": roll,
        "pitch": pitch,
    }


In [7]:
# ============================================================
# 7. Plotfunctie
# ============================================================
def plot_response_for_condition(Hs: float, Tp: float, response_without: dict, response_with: dict):
    """
    Maakt per conditie drie plots: heave, roll en pitch.
    """
    dof_info = {
        "heave": {"label": "Heave", "unit": "m"},
        "roll":  {"label": "Roll",  "unit": "deg"},
        "pitch": {"label": "Pitch", "unit": "deg"},
    }

    for dof, info in dof_info.items():
        t_without = response_without["time"]
        t_with = response_with["time"]

        mask_without = t_without >= plot_start_time
        mask_with = t_with >= plot_start_time

        plt.figure(figsize=(13, 5))

        plt.plot(
            t_without[mask_without],
            response_without[dof][mask_without],
            label="Without viscous damping",
            linewidth=1.5,
        )

        plt.plot(
            t_with[mask_with],
            response_with[dof][mask_with],
            label="With viscous damping",
            linewidth=1.5,
        )

        plt.xlabel("Time [s]")
        plt.ylabel(f"{info['label']} [{info['unit']}]")
        plt.title(f"{info['label']} response - Hs = {Hs:g} m, Tp = {Tp:g} s")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
# ============================================================
# 8. Alle golfcondities runnen
# ============================================================
all_responses = {}

for _, row in wave_conditions.iterrows():
    Hs = float(row["Hs"])
    Tp = float(row["Tp"])

    condition_key = f"Hs{Hs:g}_Tp{Tp:g}"

    print("=" * 80)
    print(f"Running wave condition: Hs = {Hs:g} m, Tp = {Tp:g} s")
    print("=" * 80)

    print("Running WITHOUT viscous damping...")
    model_without = run_orcaflex_wave_simulation(
        model_path=model_path,
        Hs=Hs,
        Tp=Tp,
        use_viscous_damping=False,
    )
    response_without = get_wave_response(model_without)

    print("Running WITH viscous damping...")
    model_with = run_orcaflex_wave_simulation(
        model_path=model_path,
        Hs=Hs,
        Tp=Tp,
        use_viscous_damping=True,
    )
    response_with = get_wave_response(model_with)

    all_responses[condition_key] = {
        "Hs": Hs,
        "Tp": Tp,
        "without_viscous_damping": response_without,
        "with_viscous_damping": response_with,
    }

    plot_response_for_condition(
        Hs=Hs,
        Tp=Tp,
        response_without=response_without,
        response_with=response_with,
    )

print("Klaar.")


NameError: name 'wave_conditions' is not defined